In [1]:
import numpy as np
import time 
import optax
import sys 
import os 
import jax
import jax.numpy as jnp
from jax import value_and_grad, vmap, jit
from sklearn.metrics import mean_squared_error

from openmm.app import PDBFile
from openmm.unit import angstrom
from openmm.app import CutoffPeriodic
from functools import partial
import pickle

from dmff.api import Hamiltonian
from dmff.utils import jit_condition
from dmff.common import nblist

# from tools import *

from jax import config
# config.update("jax_enable_x64", True)
config.update("jax_debug_nans", True)  # Enable NaN checking

import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
import scienceplots
plt.style.use(['science','no-latex'])
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 200

os.environ['MPLCONFIGDIR'] = os.getcwd() + "/configs/"

/opt/anaconda3/envs/jaxmd/lib/python3.10/site-packages/dmff/common/nblist.py:17: UserWarning: WARNING: dpdpnblist not installed, users need to create neighbor list by themselves.
  warnings.warn("WARNING: dpdpnblist not installed, users need to create neighbor list by themselves.")


In [2]:

class BasePairs:
    def __init__(self, ff, pdb, pdb_A, pdb_B):
        pdb = PDBFile(pdb)
        pdb_A = PDBFile(pdb_A)
        pdb_B = PDBFile(pdb_B)
        self.H = Hamiltonian(ff)
        self.pots = self.H.createPotential(pdb.topology, nonbondedCutoff=25*angstrom, nonbondedMethod=CutoffPeriodic, ethresh=1e-4)
        self.pots_A = self.H.createPotential(pdb_A.topology, nonbondedCutoff=25*angstrom, nonbondedMethod=CutoffPeriodic, ethresh=1e-4)
        self.pots_B = self.H.createPotential(pdb_B.topology, nonbondedCutoff=25*angstrom, nonbondedMethod=CutoffPeriodic, ethresh=1e-4)

        self.pos = jnp.array(pdb.positions._value)
        self.pos_A = jnp.array(pdb_A.positions._value) 
        self.pos_B = jnp.array(pdb_B.positions._value)

        self.box = jnp.eye(3) * 6
        self.rc = 2.5
        self.nblist = nblist.NeighborList(self.box, self.rc, self.pots.meta['cov_map'])
        self.nblist_A = nblist.NeighborList(self.box, self.rc, self.pots_A.meta['cov_map'])
        self.nblist_B = nblist.NeighborList(self.box, self.rc, self.pots_B.meta['cov_map'])
        self.nblist.allocate(self.pos)
        self.nblist_A.allocate(self.pos_A)
        self.nblist_B.allocate(self.pos_B)
        self.pairs = self.nblist.pairs
        self.pairs_A = self.nblist_A.pairs
        self.pairs_B = self.nblist_B.pairs
        self.pairs_AB = self.pairs[self.pairs[:, 0] < self.pairs[:, 1]]
        self.pairs_A = self.pairs_A[self.pairs_A[:, 0] < self.pairs_A[:, 1]]
        self.pairs_B = self.pairs_B[self.pairs_B[:, 0] < self.pairs_B[:, 1]]

        self.potentials_names = ['ex', 'sr_es', 'sr_pol', 'sr_disp', 'dhf', 'dmp_es', 'dmp_disp']
        self.potentials_mapping = {
            'ex': 'SlaterExForce',
            'sr_es': 'SlaterSrEsForce',
            'sr_pol': 'SlaterSrPolForce',
            'sr_disp': 'SlaterSrDispForce',
            'dhf': 'SlaterDhfForce',
            'dmp_es': 'QqTtDampingForce',
            'dmp_disp': 'SlaterDampingForce',
        }
        
        for potentials_name in self.potentials_names:
            setattr(self, f'pots_{potentials_name}', self.pots.dmff_potentials[self.potentials_mapping[potentials_name]])
            setattr(self, f'pots_{potentials_name}_A', self.pots_A.dmff_potentials[self.potentials_mapping[potentials_name]])
            setattr(self, f'pots_{potentials_name}_B', self.pots_B.dmff_potentials[self.potentials_mapping[potentials_name]])

    def cal_E(self, params0, pos_A, pos_B):
        params = params_convert(params0)
        # get position array
        pos_A *= 0.1
        pos_B *= 0.1
        pos_AB = jnp.concatenate([pos_A, pos_B], axis=0)
        box = self.box
        #####################
        # exchange repulsion
        #####################
        E_ex = self.pots_ex(pos_AB, box, self.pairs_AB, params)\
               - self.pots_ex_A(pos_A, box, self.pairs_A, params)\
               - self.pots_ex_B(pos_B, box, self.pairs_B, params)

        #######################
        # electrostatic
        #######################
        E_dmp_es = self.pots_dmp_es(pos_AB, box, self.pairs_AB, params) \
                    - self.pots_dmp_es_A(pos_A, box, self.pairs_A, params) \
                    - self.pots_dmp_es_B(pos_B, box, self.pairs_B, params)
        E_sr_es = self.pots_sr_es(pos_AB, box, self.pairs_AB, params) \
                - self.pots_sr_es_A(pos_A, box, self.pairs_A, params) \
                - self.pots_sr_es_B(pos_B, box, self.pairs_B, params)

        ###################################
        # polarization (induction) energy
        ###################################
        E_sr_pol = self.pots_sr_pol(pos_AB, box, self.pairs_AB, params) \
                    - self.pots_sr_pol_A(pos_A, box, self.pairs_A, params) \
                    - self.pots_sr_pol_B(pos_B, box, self.pairs_B, params)

        #############
        # dispersion
        #############
        E_dmp_disp = self.pots_dmp_disp(pos_AB, box, self.pairs_AB, params) \
                    - self.pots_dmp_disp_A(pos_A, box, self.pairs_A, params) \
                    - self.pots_dmp_disp_B(pos_B, box, self.pairs_B, params)
        E_sr_disp = self.pots_sr_disp(pos_AB, box, self.pairs_AB, params) \
                    - self.pots_sr_disp_A(pos_A, box, self.pairs_A, params) \
                    - self.pots_sr_disp_B(pos_B, box, self.pairs_B, params)

        ###########
        # dhf
        ###########
        E_dhf = self.pots_dhf(pos_AB, box, self.pairs_AB, params) \
                 - self.pots_dhf_A(pos_A, box, self.pairs_A, params) \
                 - self.pots_dhf_B(pos_B, box, self.pairs_B, params)

        E_es = E_dmp_es + E_sr_es
        E_pol = E_sr_pol
        E_disp = E_dmp_disp + E_sr_disp
        E_tot = E_ex + E_es + E_pol + E_disp + E_dhf
        return E_ex, E_es, E_pol, E_disp, E_dhf, E_tot 


In [3]:

# return data needed to fitting
def get_all_relavent_key(data, arr):
    dimer_test = []
    for key in data:
        a, b = key.split('_')[-2:]
        if a in arr or b in arr:
            dimer_test.append(key)
        elif b in arr or a in arr:
            dimer_test.append(key)
        else:
            continue
    return dimer_test

def get_all_contain_key(data, arr):
    dimer_test = []
    for key in data:
        a, b = key.split('_')[-2:]
        if a in arr and b in arr:
            dimer_test.append(key)
        elif b in arr and a in arr:
            dimer_test.append(key)
        else:
            continue
    return dimer_test

def get_all_homo_key(data, arr):
    dimer_test = []
    for key in data:
        a, b = key.split('_')[-2:]
        if a == b and b in arr:
            dimer_test.append(key)
        else:
            continue
    return dimer_test

# return data needed to fitting
def get_data_key(data, salt, solvent):
    dimer_test = []
    for key in data:
        a, b = key.split('_')[-2:]
        if a == salt and b in solvent:
            dimer_test.append(key)
        elif b == salt and a in solvent:
            dimer_test.append(key)
        else:
            continue
    return dimer_test

In [4]:
def calculate_rmsd(energy, energy_ref):
    energy = np.array(energy).T
    energy_ref = np.array(energy_ref).T
    rmsd_values = [np.sqrt(np.average((energy[i] - energy_ref[i])**2)) for i in range(len(energy))]
    return rmsd_values

def check_output_decompose(dimer_train, params):
    energies_pred = []
    energies_ref = []
    energy = []
    energy_ref = []
    shift = []
    # params = get_params(save_model, params0)
    for key in dimer_train:
        for sid in data[key].keys():
            scan_res = data[key][sid]
            tot_full = scan_res['tot_full']
            weights_pts = scan_res['wts']
            pos_A = jnp.array(scan_res['posA'])
            pos_B = jnp.array(scan_res['posB'])
            E_ex, E_es, E_pol, E_disp, E_dhf, E_tot = cal_energy[key](params, pos_A, pos_B)
            scan_res['sr_ex'], \
            scan_res['sr_es'], \
            scan_res['sr_pol'], \
            scan_res['sr_disp'], \
            scan_res['sr_dhf'],\
            scan_res['sr_ff'] = E_ex, E_es, E_pol, E_disp, E_dhf, E_tot
            E_ref = scan_res
            npts = len(E_tot)
            for ipt in range(npts):
                # if weights_pts[ipt] > 1e-2:
                # if tot_full[ipt] < 25:
                energy.append([E_tot[ipt],E_ex[ipt],E_es[ipt],E_pol[ipt],E_disp[ipt],E_dhf[ipt]])
                energy_ref.append([E_ref['tot'][ipt],E_ref['ex'][ipt],E_ref['es'][ipt],E_ref['pol'][ipt],E_ref['disp'][ipt],E_ref['dhf'][ipt]])
                shift.append(scan_res['shift'][ipt])
                if tot_full[ipt] < 25:
                    energies_pred.append([E_tot[ipt],E_ex[ipt],E_es[ipt],E_pol[ipt],E_disp[ipt],E_dhf[ipt]])
                    energies_ref.append([E_ref['tot'][ipt],E_ref['ex'][ipt],E_ref['es'][ipt],E_ref['pol'][ipt],E_ref['disp'][ipt],E_ref['dhf'][ipt]])
        rmsd_values = calculate_rmsd(energy, energy_ref)
        rmsd_midrange_value = calculate_rmsd(energies_pred, energies_ref)
        # print(key, '%.3f'%rmsd_values[0])
    return np.array(energy), np.array(energy_ref), np.array(rmsd_values), np.array(rmsd_midrange_value), np.array(shift)


def check_output_wb97(dimer_train, params):
    energies_pred = []
    energies_ref = []
    energy = []
    energy_ref = []
    shift = []
    # params = get_params(save_model, params0)
    for key in dimer_train:
        for sid in data[key].keys():
            scan_res = data[key][sid]
            tot_full = scan_res['tot_full']
            weights_pts = scan_res['wts']
            pos_A = jnp.array(scan_res['posA'])
            pos_B = jnp.array(scan_res['posB'])
            E_ex, E_es, E_pol, E_disp, E_dhf, E_tot = cal_energy[key](params, pos_A, pos_B)
            scan_res['sr_ex'], \
            scan_res['sr_es'], \
            scan_res['sr_pol'], \
            scan_res['sr_disp'], \
            scan_res['sr_dhf'],\
            scan_res['sr_ff'] = E_ex, E_es, E_pol, E_disp, E_dhf, E_tot
            E_ref = scan_res
            npts = len(E_tot)
            for ipt in range(npts):
                # if weights_pts[ipt] > 1e-2:
                # if tot_full[ipt] < 25:
                energy.append([E_tot[ipt]])
                energy_ref.append([E_ref['tot'][ipt]])
                shift.append(scan_res['shift'][ipt])
                if tot_full[ipt] < 25:
                    energies_pred.append([E_tot[ipt]])
                    energies_ref.append([E_ref['tot'][ipt]])
        rmsd_values = calculate_rmsd(energy, energy_ref)
        rmsd_midrange_value = calculate_rmsd(energies_pred, energies_ref)
        # print(key, '%.3f'%rmsd_values[0])
    return np.array(energy), np.array(energy_ref), np.array(rmsd_values), np.array(rmsd_midrange_value), np.array(shift)

In [5]:
# get params or restart from fitted params
def get_params(restart, params0):
    comps = ['ex', 'es', 'pol', 'disp', 'dhf', 'tot']
    if restart is None:
        params = {}
        sr_forces = {
                'ex': 'SlaterExForce',
                'es': 'SlaterSrEsForce',
                'pol': 'SlaterSrPolForce',
                'disp': 'SlaterSrDispForce',
                'dhf': 'SlaterDhfForce',
                }
        for k in params0['ADMPPmeForce']:
            params[k] = params0['ADMPPmeForce'][k]
        for k in params0['ADMPDispPmeForce']:
            params[k] = params0['ADMPDispPmeForce'][k]
        for c in comps:
            if c == 'tot':
                continue
            force = sr_forces[c]
            for k in params0[sr_forces[c]]:
                if k == 'A':
                    params['A_'+c] = params0[sr_forces[c]][k]
                else:
                    params[k] = params0[sr_forces[c]][k]
        # a random initialization of A
        for c in comps:
            if c == 'tot':
                continue
            params['A_'+c] = jnp.array(np.random.random(params['A_'+c].shape))
        # specify charges for es damping
        params['Q'] = params0['QqTtDampingForce']['Q']
    else:
        with open(restart, 'rb') as ifile:
            params = pickle.load(ifile)
    return params


# get params or restart from fitted params
def get_params_init(params0):
    comps = ['ex', 'es', 'pol', 'disp', 'dhf', 'tot']
    params = {}
    sr_forces = {
            'ex': 'SlaterExForce',
            'es': 'SlaterSrEsForce',
            'pol': 'SlaterSrPolForce',
            'disp': 'SlaterSrDispForce',
            'dhf': 'SlaterDhfForce',
            }
    for k in params0['ADMPPmeForce']:
        params[k] = params0['ADMPPmeForce'][k]
    for k in params0['ADMPDispPmeForce']:
        params[k] = params0['ADMPDispPmeForce'][k]
    for c in comps:
        if c == 'tot':
            continue
        force = sr_forces[c]
        for k in params0[sr_forces[c]]:
            if k == 'A':
                params['A_'+c] = params0[sr_forces[c]][k]
            else:
                params[k] = params0[sr_forces[c]][k]
    # a random initialization of A
    # for c in comps:
    #     if c == 'tot':
    #         continue
    #     params['A_'+c] = jnp.array(np.random.random(params['A_'+c].shape))
    # specify charges for es damping
    params['Q'] = params0['QqTtDampingForce']['Q']
    return params
    
def params_convert(params):
    params_ex = {}
    params_sr_es = {}
    params_sr_pol = {}
    params_sr_disp = {}
    params_dhf = {}
    params_dmp_es = {}  # electrostatic damping
    params_dmp_disp = {} # dispersion damping
    for k in ['B']:
        params_ex[k] = params[k]
        params_sr_es[k] = params[k]
        params_sr_pol[k] = params[k]
        params_sr_disp[k] = params[k]
        params_dhf[k] = params[k]
        params_dmp_es[k] = params[k]
        params_dmp_disp[k] = params[k]
    if 'C' in params:
        for k in ['C']:
            params_ex[k] = params[k]
    if 'D' in params:
        for k in ['D']:
            params_ex[k] = params[k]
    params_ex['A'] = params['A_ex']
    params_sr_es['A'] = params['A_es']
    params_sr_pol['A'] = params['A_pol']
    params_sr_disp['A'] = params['A_disp']
    params_dhf['A'] = params['A_dhf']
    # damping parameters
    params_dmp_es['Q'] = params['Q']
    params_dmp_disp['C6'] = params['C6']
    params_dmp_disp['C8'] = params['C8']
    params_dmp_disp['C10'] = params['C10']
    p = {}
    p['SlaterExForce'] = params_ex
    p['SlaterSrEsForce'] = params_sr_es
    p['SlaterSrPolForce'] = params_sr_pol
    p['SlaterSrDispForce'] = params_sr_disp
    p['SlaterDhfForce'] = params_dhf
    p['QqTtDampingForce'] = params_dmp_es
    p['SlaterDampingForce'] = params_dmp_disp
    return p


In [6]:
def get_all_homo_key(data, arr):
    dimer_test = []
    for key in data:
        a, b = key.split('_')[-2:]
        if a == b and b in arr:
            dimer_test.append(key)
        else:
            continue
    return dimer_test

def get_all_contain_key(data, arr):
    dimer_test = []
    for key in data:
        a, b = key.split('_')[-2:]
        if a in arr and b in arr:
            dimer_test.append(key)
        else:
            continue
    return dimer_test

def get_either_contain_key(data, arr):
    dimer_test = []
    for key in data:
        a, b = key.split('_')[-2:]
        if a in arr or b in arr:
            dimer_test.append(key)
        else:
            continue
    return dimer_test


In [7]:
# data_file = 'data_train_final_wt_lr.pickle'
data_file = 'data_train_final_wt_lr_update_FSI.pickle'

with open(data_file, 'rb') as ifile:
    data = pickle.load(ifile)

In [8]:
@jit
def calculate_weights(E_tot_full, thresh):
    kT = 2.494  # 300 K = 2.494 kJ/mol
    weights_pts = jnp.piecewise(E_tot_full, [E_tot_full<thresh, E_tot_full>=thresh], [lambda x: jnp.array(1.0), lambda x: jnp.exp(-(x-thresh)/kT)])
    return weights_pts

ions = ['Li', 'Na', 'PF6', 'BOB', 'FSI', 'TFSI', 'BF4', 'DFP', 'DFOB']
dimer_repulsive = get_all_homo_key(data, ions)

# for pair in data.keys():
#     for sid in data[pair].keys():
#         data[pair][sid]['wts'] = jnp.ones(12)

# for pair in data.keys():
#     print(pair)
#     if pair in dimer_repulsive:
#         for sid in data[pair].keys():
#             data[pair][sid]['wts'] = jnp.ones(12)
#     else:        
#         for sid in data[pair].keys():
#             scan_res = data[pair][sid]
#             E_tot_full = scan_res['tot_full']
#             thresh = 25
#             data[pair][sid]['wts'] = calculate_weights(E_tot_full, thresh)

for pair in data.keys():
    # print(pair)
    if pair in dimer_repulsive:
        for sid in data[pair].keys():
            data[pair][sid]['wts'] = jnp.ones(12)
    else:        
        for sid in data[pair].keys():
            scan_res = data[pair][sid]
            E_tot_full = scan_res['tot_full']
            thresh = 50 + np.min(scan_res['tot_full'])
            data[pair][sid]['wts'] = calculate_weights(E_tot_full, thresh)

In [19]:
ions = ['Li', 'Na', 'PF6', 'BOB', 'FSI', 'TFSI', 'BF4', 'DFP', 'DFOB']

arr = ['DEC', 'DFEA', 'DFEC', 'DMC', 'DME', 'DOL', 'EC', 'EMC', 'EP', \
        'FEC', 'FEMC', 'GBL', 'PC', 'PP', 'PS', 'SL', \
            'BF4', 'BOB', 'DFOB', 'DFP', 'FSI', 'PF6', 'TFSI']


arr = ['DEC', 'DFEA', 'DFEC', 'DMC', 'DME', 'DOL', 'EC', 'EMC', 'EP', \
        'FEC', 'FEMC', 'GBL', 'PC', 'PP', 'PS', 'SL']
# arr = ['BF4', 'BOB', 'DFOB', 'DFP', 'FSI', 'PF6', 'TFSI', 'Li', 'Na']
arr1 = ['CN1', 'CN2']

dimer_test_homo = get_all_homo_key(data, ions)
dimer_test_all = list(data.keys())
# dimer_test_contain = get_either_contain_key(data, arr)
dimer_test_contain1 = get_either_contain_key(data, arr1)
dimer_test_contain2 = get_all_contain_key(data, arr)
dimer_test_contain3 = get_either_contain_key(data, ions)

# dimer_test_contain1 = get_all_homo_key(data, ions)

dimer_train = sorted(list(set(dimer_test_all) - set(dimer_test_contain1) - set(dimer_test_contain2))) # just salts
# dimer_train = sorted(list(set(dimer_test_all) - set(dimer_test_contain1))) # all 
# dimer_train = sorted(list(set(dimer_test_contain3) - set(dimer_test_contain1) - set(dimer_test_homo))) # just anion

# dimer_train = sorted(list(set(dimer_test_contain2))) # just solvents

# dimer_train = ['conf_002_DME_DME', ]
dimer_train = get_all_homo_key(data, ['DME', 'PS'])
dimer_train = get_all_contain_key(data, ['Li', 'PF6','DEC'])

print(len(dimer_train),dimer_train)
dimer_train.sort()
print(dimer_train)


6 ['conf_000_DEC_DEC', 'conf_045_Li_Li', 'conf_047_PF6_PF6', 'conf_051_Li_PF6', 'conf_059_Li_DEC', 'conf_077_PF6_DEC']
['conf_000_DEC_DEC', 'conf_045_Li_Li', 'conf_047_PF6_PF6', 'conf_051_Li_PF6', 'conf_059_Li_DEC', 'conf_077_PF6_DEC']


In [20]:
data.keys()

dict_keys(['conf_000_DEC_DEC', 'conf_001_DMC_DMC', 'conf_002_DME_DME', 'conf_003_EC_EC', 'conf_004_EMC_EMC', 'conf_005_FEC_FEC', 'conf_006_PC_PC', 'conf_007_PP_PP', 'conf_008_PS_PS', 'conf_009_DEC_DMC', 'conf_010_DEC_DME', 'conf_011_DEC_EC', 'conf_012_DEC_EMC', 'conf_013_DEC_FEC', 'conf_014_DEC_PC', 'conf_015_DEC_PP', 'conf_016_DEC_PS', 'conf_017_DMC_DME', 'conf_018_DMC_EC', 'conf_019_DMC_EMC', 'conf_020_DMC_FEC', 'conf_021_DMC_PC', 'conf_022_DMC_PP', 'conf_023_DMC_PS', 'conf_024_DME_EC', 'conf_025_DME_EMC', 'conf_026_DME_FEC', 'conf_027_DME_PC', 'conf_028_DME_PP', 'conf_029_DME_PS', 'conf_030_EC_EMC', 'conf_031_EC_FEC', 'conf_032_EC_PC', 'conf_033_EC_PP', 'conf_034_EC_PS', 'conf_035_EMC_FEC', 'conf_036_EMC_PC', 'conf_037_EMC_PP', 'conf_038_EMC_PS', 'conf_039_FEC_PC', 'conf_040_FEC_PP', 'conf_041_FEC_PS', 'conf_042_PC_PP', 'conf_043_PC_PS', 'conf_044_PP_PS', 'conf_048_BOB_BOB', 'conf_049_FSI_FSI', 'conf_086_BOB_DEC', 'conf_087_BOB_DMC', 'conf_088_BOB_DME', 'conf_089_BOB_EC', 'conf_090_

In [21]:
# get params or restart from fitted params
def get_params(restart, params0):
    comps = ['ex', 'es', 'pol', 'disp', 'dhf', 'tot']
    if restart is None:
        params = {}
        sr_forces = {
                'ex': 'SlaterExForce',
                'es': 'SlaterSrEsForce',
                'pol': 'SlaterSrPolForce',
                'disp': 'SlaterSrDispForce',
                'dhf': 'SlaterDhfForce',
                }
        for k in params0['ADMPPmeForce']:
            params[k] = params0['ADMPPmeForce'][k]
        for k in params0['ADMPDispPmeForce']:
            params[k] = params0['ADMPDispPmeForce'][k]
        for c in comps:
            if c == 'tot':
                continue
            force = sr_forces[c]
            for k in params0[sr_forces[c]]:
                if k == 'A':
                    params['A_'+c] = params0[sr_forces[c]][k]
                else:
                    params[k] = params0[sr_forces[c]][k]
        # a random initialization of A
        for c in comps:
            if c == 'tot':
                continue
            params['A_'+c] = jnp.array(np.random.random(params['A_'+c].shape)) * 100
        # specify charges for es damping
        params['Q'] = params0['QqTtDampingForce']['Q']
    else:
        with open(restart, 'rb') as ifile:
            params = pickle.load(ifile)
    return params



In [22]:
restart = None
# ff = 'output_xml/output.xml'
ff = 'output_xml/output.B_pol.xml'
params0 = Hamiltonian(ff).getParameters()
params = get_params(restart, params0)
# params = get_params_init(params0)

In [23]:
params['B_pol'].shape

(157,)

In [24]:
class_instances = {}
cal_energy = {}    
MSELoss_grad = {} 

dimer_train.sort()
for pair in dimer_train: 
    if pair not in MSELoss_grad:
        print(pair)
        conf, numb_conf, monomer_A, monomer_B = pair.split('_')
        dimer_file = f'dimer_{numb_conf}_{monomer_A}_{monomer_B}'
        dimer_file = f'dimer_bank/{dimer_file}.pdb'
        pdb_A_file = f'pdb_bank/{monomer_A}.pdb'
        pdb_B_file = f'pdb_bank/{monomer_B}.pdb'
        class_instances[pair] = BasePairs(ff, dimer_file, pdb_A_file, pdb_B_file)
for class_name, class_instance in class_instances.items():
    cal_energy[class_name] = jit(vmap(class_instance.cal_E, in_axes=(None, 0, 0), out_axes=(0, 0, 0, 0, 0, 0)))

for key in dimer_train:
    batch = list(data[key].keys())[1]
    if key not in MSELoss_grad:
        # def MSELoss(params, data):
        #     '''
        #     The weighted mean squared error loss function
        #     Conducted for each scan
        #     '''
        #     # batch = padding(batch)
        #     scan_res = data
        #     comps = ['ex', 'es', 'pol', 'disp', 'dhf', 'tot']
        #     weights_comps = jnp.array([0.1, 0.1, 0.1, 0.1, 0.1, 1.0])
        #     weights_pts = scan_res['wts']
        #     npts = len(weights_pts)

        #     energies = {
        #             'ex': jnp.zeros(npts),
        #             'es': jnp.zeros(npts),
        #             'pol': jnp.zeros(npts),
        #             'disp': jnp.zeros(npts),
        #             'dhf': jnp.zeros(npts),
        #             'tot': jnp.zeros(npts)
        #             }

        #     E_ex, E_es, E_pol, E_disp, E_dhf, E_tot = cal_energy[key](params, scan_res['posA'], scan_res['posB'])
            
        #     for ipt in range(npts):
        #         energies['ex'] = energies['ex'].at[ipt].set(E_ex[ipt])
        #         energies['es'] = energies['es'].at[ipt].set(E_es[ipt])
        #         energies['pol'] = energies['pol'].at[ipt].set(E_pol[ipt])
        #         energies['disp'] = energies['disp'].at[ipt].set(E_disp[ipt])
        #         energies['dhf'] = energies['dhf'].at[ipt].set(E_dhf[ipt])
        #         energies['tot'] = energies['tot'].at[ipt].set(E_tot[ipt])

        #     errs = jnp.zeros(len(comps))
        #     for ic, c in enumerate(comps):
        #         dE = scan_res[c] - energies[c] 
        #         mse = dE**2 * weights_pts / jnp.sum(weights_pts)
        #         errs = errs.at[ic].set(jnp.sum(mse))
        #     loss = jnp.sum(weights_comps * errs)
        #     return loss


        def MSELoss(params, data):
            '''
            Asymmetric Huber Loss
            Combines the robustness of Huber Loss with an asymmetric penalty to fix underestimation.
            '''
            # --- 1. 数据准备 ---
            scan_res = data
            comps = ['ex', 'es', 'pol', 'disp', 'dhf', 'tot']
            # 可以根据需要调整不同能量分量的权重
            weights_comps = jnp.array([0.1, 0.1, 0.1, 0.1, 0.1, 0.01])
            weights_pts = scan_res['wts']
            npts = len(weights_pts)

            # --- 2. 超参数设置 ---
            # delta: Huber Loss 的阈值。
            # 误差绝对值 < delta 时是 MSE，> delta 时是 MAE。
            # 建议设为数据平均误差的某个百分比，或者直接设为 0.1 ~ 1.0 左右尝试。
            delta = 1.0 
            
            # alpha: 低估惩罚系数。
            # alpha > 1.0 表示对"低估"惩罚更重。
            # 如果预测值总是偏低，建议从 2.0 开始尝试，不够就加到 5.0 或 10.0。
            alpha = 5.0 

            # --- 3. 计算预测能量 ---
            # (假设 cal_energy 已经定义好了)
            energies = {
                    'ex': jnp.zeros(npts),
                    'es': jnp.zeros(npts),
                    'pol': jnp.zeros(npts),
                    'disp': jnp.zeros(npts),
                    'dhf': jnp.zeros(npts),
                    'tot': jnp.zeros(npts)
                    }

            # 注意：这里需要您保证 cal_energy[key] 可用，或者替换为您实际的函数调用
            E_ex, E_es, E_pol, E_disp, E_dhf, E_tot = cal_energy[key](params, scan_res['posA'], scan_res['posB'])
            
            # 填充 energies 字典
            for ipt in range(npts):
                energies['ex'] = energies['ex'].at[ipt].set(E_ex[ipt])
                energies['es'] = energies['es'].at[ipt].set(E_es[ipt])
                energies['pol'] = energies['pol'].at[ipt].set(E_pol[ipt])
                energies['disp'] = energies['disp'].at[ipt].set(E_disp[ipt])
                energies['dhf'] = energies['dhf'].at[ipt].set(E_dhf[ipt])
                energies['tot'] = energies['tot'].at[ipt].set(E_tot[ipt])

            # --- 4. 计算 Loss ---
            errs = jnp.zeros(len(comps))
            
            for ic, c in enumerate(comps):
                # 计算残差: 预测 - 目标
                # diff < 0 代表低估，diff > 0 代表高估
                diff = energies[c] - scan_res[c]
                abs_diff = jnp.abs(diff)
                
                # A. 标准 Huber 计算部分
                # quadratic: 平方部分 (0.5 * x^2)
                # linear: 线性部分 (delta * (|x| - 0.5 * delta))
                is_small_error = abs_diff < delta
                squared_loss = 0.5 * diff**2
                linear_loss = delta * (abs_diff - 0.5 * delta)
                
                # 组合得到基础 Huber Loss
                base_huber = jnp.where(is_small_error, squared_loss, linear_loss)
                
                # B. 非对称惩罚部分
                # 如果 diff < 0 (低估)，权重乘以 alpha；否则权重为 1.0
                penalty_mask = jnp.where(diff < 0, alpha, 1.0)
                
                # C. 最终加权计算
                weighted_loss = base_huber * penalty_mask * weights_pts / jnp.sum(weights_pts)
                errs = errs.at[ic].set(jnp.sum(weighted_loss))

            loss = jnp.sum(weights_comps * errs)
            return loss
    
    MSELoss_grad[key] = jit(value_and_grad(MSELoss, argnums=(0)))
    err, gradients = MSELoss_grad[key](params, data[key][batch])
    print(key , err)



conf_000_DEC_DEC
conf_045_Li_Li
conf_047_PF6_PF6
conf_051_Li_PF6
conf_059_Li_DEC
conf_077_PF6_DEC
conf_000_DEC_DEC 17.27415302029445
conf_045_Li_Li 2.02487981296036
conf_047_PF6_PF6 402.04958746397585
conf_051_Li_PF6 350.9779189752405
conf_059_Li_DEC 64.85231067909433
conf_077_PF6_DEC 117.17409308323073


In [26]:
trunk = []
for key in dimer_train:
    for batch in data[key]:
        trunk.append([key,batch])

os.makedirs('params', exist_ok=True)
# save_model = 'params/params.solvents.penalty.fix43.Aex.salts.pickle'
save_model = 'params/params.dampingtest.pickle'


# weights_comps = jnp.array([0.1, 0.1, 0.1, 0.1, 0.1, 1.0])
comps = ['ex', 'es', 'pol', 'disp', 'dhf', 'tot']

def mask_fn(grads):
    for k in grads:
        if k.startswith('A_') or k == 'B' or k == 'B_pol':
            continue
        else:
            grads[k] = 0.0
    return grads

lr = 0.1
optimizer = optax.adam(lr)
opt_state = optimizer.init(params)
n_epochs = 1000

loss_train = []
loss_test = []
for i_epoch in range(n_epochs):
    np.random.shuffle(trunk)
    for key0, batch in trunk:
        loss, grads = MSELoss_grad[key0](params, data[key0][batch])
        grad = mask_fn(grads)
        updates, opt_state = optimizer.update(grad, opt_state)
        params = optax.apply_updates(params, updates)

    print(f"{i_epoch} {loss:.6f} {key0}")
        
    if (i_epoch % 10 == 0):
        with open(save_model, 'wb') as ofile:
            pickle.dump(params, ofile)

KeyboardInterrupt: 

In [33]:
# 初始化字典
class_instances = {}
cal_energy = {}
MSELoss_grad = {} 

# 处理 dimer 训练数据并创建实例
for pair in dimer_train: 
    if pair not in MSELoss_grad:
        print(pair)
        conf, numb_conf, monomer_A, monomer_B = pair.split('_')
        dimer_file = f'dimer_bank/dimer_{numb_conf}_{monomer_A}_{monomer_B}.pdb'
        pdb_A_file = f'pdb_bank/{monomer_A}.pdb'
        pdb_B_file = f'pdb_bank/{monomer_B}.pdb'
        class_instances[pair] = BasePairs(ff, dimer_file, pdb_A_file, pdb_B_file)

# 向量化能量计算函数
for class_name, class_instance in class_instances.items():
    cal_energy[class_name] = jit(
        vmap(class_instance.cal_E, 
             in_axes=(None, 0, 0), 
             out_axes=(0, 0, 0, 0, 0, 0))
    )

weights_comps = jnp.array([0.1, 0.1, 0.1, 0.1, 0.1, 1.0])
# 定义惩罚掩码：ex(0)和tot(5)项为1，其他为0
penalty_mask = jnp.array([1.0, 0.0, 0.0, 0.0, 0.0, 1.0])

# 为每个键定义损失函数和梯度
for key in dimer_train:
    batch = list(data[key].keys())[1]
    
    if key not in MSELoss_grad:
        @jit
        def MSELoss(params, data, weights_comps, penalty_mask, penalty_scale=10.0):
            """完全向量化的损失函数，无显式循环和条件判断"""
            scan_res = data
            weights_pts = scan_res['wts']
            weights_sum = jnp.sum(weights_pts)
            
            # 计算所有能量分量并堆叠为数组
            energies = jnp.stack(cal_energy[key](params, scan_res['posA'], scan_res['posB']))
            
            # 提取目标值并堆叠为数组（与energies结构对应）
            targets = jnp.stack([
                scan_res['ex'], scan_res['es'], scan_res['pol'],
                scan_res['disp'], scan_res['dhf'], scan_res['tot']
            ])
            
            # 计算所有分量的误差 (能量预测 - 目标值)
            dE = energies - targets
            
            # 计算加权MSE (完全向量化)
            mse = jnp.sum((dE**2 * weights_pts) / weights_sum, axis=1)
            total_err = jnp.sum(weights_comps * mse)
            
            # 计算惩罚项 (仅对ex和tot应用，通过掩码实现)
            under_pred = jnp.maximum(-dE, 0.0)  # 等同于 max(target - energy, 0)
            penalty = penalty_scale * jnp.sum(
                penalty_mask * jnp.sum((under_pred**2 * weights_pts), axis=1)
            )
            
            return total_err + penalty
        
        # 创建带梯度的损失函数
        MSELoss_grad[key] = jit(value_and_grad(MSELoss, argnums=(0)))
    
    # 计算误差和梯度
    err, gradients = MSELoss_grad[key](params, data[key][batch], weights_comps, penalty_mask)
    print(f"{key} {batch} {err}")
    

conf_002_DME_DME
conf_008_PS_PS
conf_002_DME_DME 020 156.7442108552361
conf_008_PS_PS 020 396.64136784465575


In [34]:
for pair in dimer_train:
    # print(pair)
    if pair in dimer_repulsive:
        for sid in data[pair].keys():
            data[pair][sid]['wts'] = jnp.ones(12)
    else:        
        for sid in data[pair].keys():
            scan_res = data[pair][sid]
            E_tot_full = scan_res['tot_full']
            thresh = 50 + np.min(scan_res['tot_full'])
            data[pair][sid]['wts'] = calculate_weights(E_tot_full, thresh)
trunk = []
for key in dimer_train:
    for batch in data[key]:
        trunk.append([key,batch])

os.makedirs('params', exist_ok=True)
# save_model = 'params/params.2.ABC.solvents.pospenalty.25.pickle'
save_model = 'params/params.penalty.DME.pickle'

weights_comps = jnp.array([0.1, 0.1, 0.1, 0.1, 0.1, 0.0])
comps = ['ex', 'es', 'pol', 'disp', 'dhf', 'tot']
# 定义惩罚掩码：ex(0)和tot(5)项为1，其他为0
penalty_mask = jnp.array([1.0, 0.0, 0.0, 0.0, 0.0, 0.0])

def mask_fn(grads):
    for k in grads:
        if k.startswith('A_') or k == 'B':
            # just solvent
            # grads[k] = grads[k].at[:130].set(0.0) # just solvent
            # grads[k] = grads[k].at[157:].set(0.0) # just solvent
            # just salt
            # grads[k] = grads[k].at[130:157].set(0.0) # just salt
            continue
        else:
            grads[k] = 0.0
    return grads

lr = 0.1
optimizer = optax.adam(lr)
opt_state = optimizer.init(params)
n_epochs = 1500

loss_train = []
loss_test = []
for i_epoch in range(n_epochs):
    np.random.shuffle(trunk)
    for key0, batch in trunk:
        loss, grads = MSELoss_grad[key0](params, data[key0][batch], weights_comps, penalty_mask)
        grad = mask_fn(grads)
        updates, opt_state = optimizer.update(grad, opt_state)
        params = optax.apply_updates(params, updates)
        # params = params.replace(d=jnp.maximum(params.d, 1e-32))  # 1e-32避免d=0导致后续计算问题
        # params['D'] = jnp.maximum(params['D'], 1e-32)  # 确保d非负，1e-32避免为0
        # params['C'] = jnp.maximum(params['C'], 1e-32)  # 确保d非负，1e-32避免为0

        params['A_dhf'] = jnp.maximum(params['A_dhf'], 1e-32)  # 确保非负，1e-32避免为0
        params['A_disp'] = jnp.maximum(params['A_disp'], 1e-32)  # 确保非负，1e-32避免为0
        params['A_es'] = jnp.maximum(params['A_es'], 1e-32)  # 确保非负，1e-32避免为0
        params['A_ex'] = jnp.maximum(params['A_ex'], 1e-32)  # 确保非负，1e-32避免为0
        params['A_pol'] = jnp.maximum(params['A_pol'], 1e-32)  # 确保非负，1e-32避免为0
        
    print(f"{i_epoch} {loss:.6f} {key0}")
        
    if (i_epoch % 10 == 0):
        with open(save_model, 'wb') as ofile:
            pickle.dump(params, ofile)

0 7.756803 conf_008_PS_PS
1 12.092342 conf_002_DME_DME
2 1.303349 conf_008_PS_PS
3 25.365578 conf_008_PS_PS
4 32.953863 conf_008_PS_PS
5 10.891993 conf_002_DME_DME
6 10.764365 conf_008_PS_PS
7 16.754032 conf_002_DME_DME
8 12.566402 conf_008_PS_PS
9 12.100518 conf_002_DME_DME
10 4.480273 conf_008_PS_PS
11 23.203261 conf_002_DME_DME
12 22.159427 conf_008_PS_PS
13 14.423718 conf_002_DME_DME
14 3.587849 conf_008_PS_PS
15 70.744319 conf_002_DME_DME
16 7.912660 conf_002_DME_DME
17 7.380917 conf_002_DME_DME
18 8.049134 conf_002_DME_DME
19 8.199297 conf_008_PS_PS
20 1.713093 conf_002_DME_DME
21 4.454542 conf_008_PS_PS
22 18.747328 conf_008_PS_PS
23 0.652851 conf_002_DME_DME
24 23.126160 conf_008_PS_PS
25 29.416851 conf_002_DME_DME
26 3.612089 conf_002_DME_DME
27 7.870901 conf_002_DME_DME
28 102.855400 conf_002_DME_DME
29 10.485582 conf_008_PS_PS
30 13.887208 conf_002_DME_DME
31 5.828241 conf_002_DME_DME
32 7.974338 conf_002_DME_DME
33 12.044495 conf_008_PS_PS
34 6.371719 conf_008_PS_PS
35 29.0

KeyboardInterrupt: 

# 迁移到wb97mv

In [35]:
data_file = 'data_wb97_wt_lr.pickle'
with open(data_file, 'rb') as ifile:
    data = pickle.load(ifile)


data_file = 'data_wb97_all_tot_wt_lr.pickle'
with open(data_file, 'rb') as ifile:
    data = pickle.load(ifile)

In [ ]:
# # return data needed to fitting
# def get_data_single_key(data, salt, solvent):
#     dimer_test = []
#     for key in data:
#         a, b = key.split('_')[-2:]
#         if a == salt and b in solvent:
#             dimer_test.append(key)
#         elif b == salt and a in solvent:
#             dimer_test.append(key)
#         else:
#             continue
#     return dimer_test

# with open('atype_data.pickle', 'rb') as ifile:
#     atype_data = pickle.load(ifile)

# bank = ['DEC', 'DFEA', 'DFEC', 'DMC', 'DME', 'DOL', 'EC', 'EMC', 'EP', \
#         'FEC', 'FEMC', 'GBL', 'PC', 'PP', 'PS', 'SL',\
#         'BF4', 'BOB', 'DFOB', 'DFP', 'FSI', 'PF6', 'TFSI', 'Li', 'Na','CN1','CN2']
# num_atomtype = 0
# num_atomtypes = {}    
# for key in bank:
#     start_atype = num_atomtype
#     end_atype = num_atomtype + len(atype_data[key])
#     num_atomtypes[key] = [start_atype, end_atype]
#     num_atomtype += len(atype_data[key])

# train_target = num_atomtypes[salt]
# salts = ['SL']
# solvents = ['DEC', 'DFEA', 'DFEC', 'DMC', 'DME', 'DOL', 'EC', 'EMC', 'EP', 'FEC', 'FEMC', 'GBL', 'PC', 'PP', 'PS', 'SL']# 'CN1', 'CN2']
# dimer_train = get_data_single_key(data, salt, solvents)
# print()

# def mask_fn(grads, num_atomtype):
#     for k in grads:
#         if k.startswith('A_') or k == 'B':
#             grads[k] = grads[k].at[:num_atomtype[0]].set(0.0)
#             grads[k] = grads[k].at[num_atomtype[1]:].set(0.0)
#             continue
#         else:
#             grads[k] = 0.0
#     return grads

In [ ]:
# ions = ['Li', 'Na', 'PF6', 'BOB', 'FSI', 'TFSI', 'BF4', 'DFP', 'DFOB']

# arr = ['DEC', 'DFEA', 'DFEC', 'DMC', 'DME', 'DOL', 'EC', 'EMC', 'EP', \
#         'FEC', 'FEMC', 'GBL', 'PC', 'PP', 'PS', 'SL', \
#             'BF4', 'BOB', 'DFOB', 'DFP', 'FSI', 'PF6', 'TFSI',]

# # arr = ['BF4', 'BOB', 'DFOB', 'DFP', 'FSI', 'PF6', 'TFSI', 'Li', 'Na']
# arr1 = ['CN1', 'CN2']

# dimer_test_homo = get_all_homo_key(data, ions)
# dimer_test_all = list(data.keys())
# # dimer_test_contain = get_either_contain_key(data, arr)
# dimer_test_contain1 = get_either_contain_key(data, arr1)
# dimer_test_contain2 = get_all_contain_key(data, arr)
# dimer_test_contain3 = get_either_contain_key(data, ions)

# # dimer_test_contain1 = get_all_homo_key(data, ions)

# # dimer_train = sorted(list(set(dimer_test_all) - set(dimer_test_contain1) - set(dimer_test_contain2))) # just salts
# # dimer_train = sorted(list(set(dimer_test_all) - set(dimer_test_contain1))) # all 
# # dimer_train = sorted(list(set(dimer_test_contain3) - set(dimer_test_contain1) - set(dimer_test_homo))) # just anion

# dimer_train = sorted(list(set(dimer_test_contain2))) # just solvents


# print(len(dimer_train),dimer_train)
# dimer_train.sort()
# print(dimer_train)


210 ['conf_000_DEC_DEC', 'conf_001_DMC_DMC', 'conf_002_DME_DME', 'conf_003_EC_EC', 'conf_004_EMC_EMC', 'conf_005_FEC_FEC', 'conf_006_PC_PC', 'conf_007_PP_PP', 'conf_008_PS_PS', 'conf_009_DEC_DMC', 'conf_010_DEC_DME', 'conf_011_DEC_EC', 'conf_012_DEC_EMC', 'conf_013_DEC_FEC', 'conf_014_DEC_PC', 'conf_015_DEC_PP', 'conf_016_DEC_PS', 'conf_017_DMC_DME', 'conf_018_DMC_EC', 'conf_019_DMC_EMC', 'conf_020_DMC_FEC', 'conf_021_DMC_PC', 'conf_022_DMC_PP', 'conf_023_DMC_PS', 'conf_024_DME_EC', 'conf_025_DME_EMC', 'conf_026_DME_FEC', 'conf_027_DME_PC', 'conf_028_DME_PP', 'conf_029_DME_PS', 'conf_030_EC_EMC', 'conf_031_EC_FEC', 'conf_032_EC_PC', 'conf_033_EC_PP', 'conf_034_EC_PS', 'conf_035_EMC_FEC', 'conf_036_EMC_PC', 'conf_037_EMC_PP', 'conf_038_EMC_PS', 'conf_039_FEC_PC', 'conf_040_FEC_PP', 'conf_041_FEC_PS', 'conf_042_PC_PP', 'conf_043_PC_PS', 'conf_044_PP_PS', 'conf_047_PF6_PF6', 'conf_048_BOB_BOB', 'conf_049_FSI_FSI', 'conf_050_TFSI_TFSI', 'conf_077_PF6_DEC', 'conf_078_PF6_DMC', 'conf_079_PF6

In [37]:

@jit
def calculate_weights(E_tot_full, thresh):
    kT = 2.494  # 300 K = 2.494 kJ/mol
    weights_pts = jnp.piecewise(E_tot_full, [E_tot_full<thresh, E_tot_full>=thresh], [lambda x: jnp.array(1.0), lambda x: jnp.exp(-(x-thresh)/kT)])
    return weights_pts

for pair in dimer_train:
    # print(pair)
    if pair in dimer_repulsive:
        for sid in data[pair].keys():
            data[pair][sid]['wts'] = jnp.ones(12)
    else:        
        for sid in data[pair].keys():
            scan_res = data[pair][sid]
            E_tot_full = scan_res['tot_full']
            thresh = 50 + np.min(scan_res['tot_full'])
            data[pair][sid]['wts'] = calculate_weights(E_tot_full, thresh)

In [38]:
# ff = 'output_xml/output.solvents.penalty.fix43.Aex.xml'
# save_model = 'params/params.solvents.penalty.fix43.Aex.salts.pickle'
restart = save_model
print(save_model)
params0 = Hamiltonian(ff).getParameters()
# params = get_params_init(params0)
params = get_params(restart, params0)

params/params.penalty.DME.pickle


In [39]:
class_instances = {}
cal_energy = {}
MSELoss_grad = {}

for pair in dimer_train: 
    if pair not in MSELoss_grad:
        print(pair)
        conf, numb_conf, monomer_A, monomer_B = pair.split('_')
        dimer_file = f'dimer_{numb_conf}_{monomer_A}_{monomer_B}'
        dimer_file = f'dimer_bank/{dimer_file}.pdb'
        pdb_A_file = f'pdb_bank/{monomer_A}.pdb'
        pdb_B_file = f'pdb_bank/{monomer_B}.pdb'
        class_instances[pair] = BasePairs(ff, dimer_file, pdb_A_file, pdb_B_file)
for class_name, class_instance in class_instances.items():
    cal_energy[class_name] = jit(vmap(class_instance.cal_E, in_axes=(None, 0, 0), out_axes=(0, 0, 0, 0, 0, 0)))

for key in dimer_train:
    batch = list(data[key].keys())[1]
    if key not in MSELoss_grad:
        def MSELoss(params, data):
            '''
            The mean squared error loss function for total energy only
            Conducted for each scan
            '''
            scan_res = data
            weights_pts = scan_res['wts']
            npts = len(weights_pts)
            weights_comps = jnp.array([0.1, 0.1, 0.1, 0.1, 0.1, 1.0])

            # 只计算总能量
            _, _, _, _, _, E_tot = cal_energy[key](params, scan_res['posA'], scan_res['posB'])
            
            # 只计算总能量的误差
            dE = E_tot - scan_res['tot']
            mse = dE**2 * weights_pts / jnp.sum(weights_pts)
            loss = jnp.sum(mse)
            return loss
        MSELoss_grad[key] = jit(value_and_grad(MSELoss, argnums=(0)))
    err, gradients = MSELoss_grad[key](params, data[key][batch])
    print(key, batch, err)

conf_002_DME_DME
conf_008_PS_PS
conf_002_DME_DME 002 111.29024674833978
conf_008_PS_PS 002 553.4868598080319


In [40]:
trunk = []
for key in dimer_train:
    for batch in data[key]:
        trunk.append([key,batch])

os.makedirs('params', exist_ok=True)
save_model = 'params/params.penalty.DME.Aex.pickle'

weights_comps = jnp.array([0.1, 0.1, 0.1, 0.1, 0.1, 1.0])

def mask_fn(grads):
    for k in grads:
        if k == 'A_ex':
            continue
        else:
            grads[k] = 0.0
    return grads

comps = ['ex', 'es', 'pol', 'disp', 'dhf', 'tot']
lr = 0.01
optimizer = optax.adam(lr)
opt_state = optimizer.init(params)
n_epochs = 1000

loss_train = []
loss_test = []
for i_epoch in range(n_epochs):
    np.random.shuffle(trunk)
    for key0, batch in trunk:
        loss, grads = MSELoss_grad[key0](params, data[key0][batch])
        grad = mask_fn(grads)
        updates, opt_state = optimizer.update(grad, opt_state)
        params = optax.apply_updates(params, updates)

        # params['A_dhf'] = jnp.maximum(params['A_dhf'], 1e-32)  # 确保非负，1e-32避免为0
        # params['A_disp'] = jnp.maximum(params['A_disp'], 1e-32)  # 确保非负，1e-32避免为0
        # params['A_es'] = jnp.maximum(params['A_es'], 1e-32)  # 确保非负，1e-32避免为0
        # params['A_ex'] = jnp.maximum(params['A_ex'], 1e-32)  # 确保非负，1e-32避免为0
        # params['A_pol'] = jnp.maximum(params['A_pol'], 1e-32)  # 确保非负，1e-32避免为0
        
    print(f"{i_epoch} {loss:.6f} {key0} {batch}")
        
    if (i_epoch % 10 == 0):
        with open(save_model, 'wb') as ofile:
            pickle.dump(params, ofile)

0 760.826421 conf_008_PS_PS 109
1 844.057076 conf_008_PS_PS 048
2 1685.150730 conf_008_PS_PS 006
3 17.168753 conf_002_DME_DME 123
4 8.434943 conf_002_DME_DME 012
5 135.932937 conf_008_PS_PS 070
6 7.821708 conf_002_DME_DME 016
7 9.632753 conf_002_DME_DME 108
8 37.173099 conf_008_PS_PS 064
9 8.404085 conf_008_PS_PS 150
10 12.143524 conf_002_DME_DME 022
11 1.507982 conf_002_DME_DME 139
12 16.709330 conf_008_PS_PS 036
13 30.278028 conf_008_PS_PS 113
14 6.506472 conf_008_PS_PS 055
15 108.185766 conf_008_PS_PS 034
16 3.189043 conf_002_DME_DME 036
17 0.850732 conf_002_DME_DME 138
18 7.880496 conf_008_PS_PS 058
19 24.012307 conf_008_PS_PS 009
20 5.712915 conf_008_PS_PS 039
21 34.557351 conf_002_DME_DME 072
22 5.472006 conf_002_DME_DME 157
23 0.493243 conf_008_PS_PS 110
24 3.338009 conf_008_PS_PS 095
25 26.099808 conf_008_PS_PS 017
26 4.252674 conf_008_PS_PS 120
27 2.412300 conf_008_PS_PS 106
28 5.822071 conf_002_DME_DME 082
29 2.823225 conf_008_PS_PS 054
30 4.170926 conf_008_PS_PS 137
31 1.166

KeyboardInterrupt: 

In [29]:
import xml.etree.ElementTree as ET
import numpy as np
import jax.numpy as jnp
import os
import pickle
from dmff.api import Hamiltonian
from typing import Dict, Any, Optional

def update_force_field(params_pickle: str, input_xml: str, output_dir: str = 'output_xml') -> str:
    """
    更新力场参数并生成新的XML文件
    
    Args:
        params_pickle: 参数pickle文件路径
        input_xml: 输入的力场XML文件路径
        output_dir: 输出目录，默认为'output_xml'
    
    Returns:
        str: 输出XML文件的路径
    """
    def params_convert(params: Dict) -> Dict:
        # 初始化所有力场参数字典
        force_params = {
            'SlaterExForce': {},
            'SlaterSrEsForce': {},
            'SlaterSrPolForce': {},
            'SlaterSrDispForce': {},
            'SlaterDhfForce': {},
            'QqTtDampingForce': {},
            'SlaterDampingForce': {}
        }
        
        # 设置B参数
        for force in force_params.values():
            force['B'] = params['B']
            # force['C'] = params['C']
        
        # 设置A参数
        force_params['SlaterExForce']['A'] = params['A_ex']
        force_params['SlaterSrEsForce']['A'] = params['A_es']
        force_params['SlaterSrPolForce']['A'] = params['A_pol']
        force_params['SlaterSrDispForce']['A'] = params['A_disp']
        force_params['SlaterDhfForce']['A'] = params['A_dhf']
        
        # 设置阻尼参数
        force_params['QqTtDampingForce']['Q'] = params['Q']
        force_params['SlaterDampingForce'].update({
            'C6': params['C6'],
            'C8': params['C8'],
            'C10': params['C10']
        })
        
        return force_params

    def get_params(restart: str, params0: Dict) -> Dict:
        """获取参数"""
        with open(restart, 'rb') as ifile:
            return pickle.load(ifile)

    try:
        # 创建输出目录
        os.makedirs(output_dir, exist_ok=True)
        
        # 获取初始参数
        params0 = Hamiltonian(input_xml).getParameters()
        params = get_params(params_pickle, params0)
        force_values = params_convert(params)
        
        # 读取并更新XML
        tree = ET.parse(input_xml)
        root = tree.getroot()
        
        # 更新参数
        for force_type, values in force_values.items():
            for elem in root.iter(force_type):
                for i, atom_elem in enumerate(elem.iter('Atom')):
                    for param in ['A', 'B']:
                        if param in values:
                            atom_elem.set(param, str(float(values[param][i])))
        
        # 生成输出文件名ew
        base_name = os.path.splitext(os.path.basename(params_pickle))[0]
        output_name = f"output{base_name.split('params')[-1]}.xml"
        # output_name = f"output.xml"
        output_path = os.path.join(output_dir, output_name)
        
        # 保存XML
        tree.write(output_path)
        print(f'已保存更新后的力场文件到: {output_path}')
        
        return output_path
        
    except Exception as e:
        print(f"更新力场文件时发生错误: {e}")
        raise

In [18]:
output_path = update_force_field(save_model, ff)

print(output_path)
pass

NameError: name 'update_force_field' is not defined